# ChunkRAG — Qwen2.5-7B

## Runtime Check

In [ ]:
import sys, subprocess
print(f"Python {sys.version}")
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                           "--format=csv,noheader"], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(f"GPU: {gpu_info.stdout.strip()}")
    if "A100" not in gpu_info.stdout:
        print("WARNING: Expected A100")
else:
    print("No GPU detected — set Runtime → Change runtime type → A100 GPU.")

## Install

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
USERNAME = "USERNAME"
# ─────────────────────────────────────────────────────────────────────────────

GITHUB_REPO = f"https://github.com/{USERNAME}/chunkrag-course-project.git"

import os, sys
REPO_DIR = "/content/chunkrag-course-project"

if not os.path.isdir(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

# Colab pre-installs torch, transformers, numpy, pandas, etc. — only install what's missing.
%pip install faiss-cpu rank-bm25 langchain-text-splitters sentence-transformers spacy "chonkie[semantic]>=1"

!pip install --no-deps --ignore-requires-python -e {REPO_DIR}
!python -m spacy download en_core_web_sm

# Make chunkrag importable after any kernel restart triggered by %pip install
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

print("Installation complete.")

## Google Drive

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/src")

from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/chunkrag_outputs/colab_qwen"

print(f"Results will be saved to: {OUTPUT_DIR}")

## Experiment Configuration

Two configs:
- **`COLAB_QUICK_CONFIG`** — 1 seed, 60 SQuAD + 30 HotpotQA examples.
- **`COLAB_FULL_CONFIG`** — 3 seeds, 400 SQuAD + 150 HotpotQA examples.

In [ ]:
_RETRIEVERS = [
    {"name": "dense", "type": "dense"},
]

_CHUNKERS = [
    {"name": "fixed_128",     "type": "fixed",     "chunk_size": 128, "chunk_overlap": 19},
    {"name": "fixed_256",     "type": "fixed",     "chunk_size": 256, "chunk_overlap": 38},
    {"name": "fixed_512",     "type": "fixed",     "chunk_size": 512, "chunk_overlap": 76},
    {"name": "recursive_256", "type": "recursive", "chunk_size": 256, "chunk_overlap": 38},
    {"name": "sentence_256",  "type": "sentence",  "chunk_size": 256},
    {"name": "semantic_256",  "type": "semantic",  "chunk_size": 256,
     "similarity_threshold": 0.72, "min_chunk_tokens": 96},
]


_BASE = {
    "embedding_model":             "sentence-transformers/all-MiniLM-L6-v2",
    "generator_model":             "Qwen/Qwen2.5-7B-Instruct",
    "generator_torch_dtype":       "bfloat16",
    "generator_use_device_map":    True,
    "device":                      "cuda",
    "retrieval_top_k":             4,
    "max_new_tokens":              48,
    "embedding_batch_size":        64,
    "generation_max_input_tokens": 1536,
    "run_parametric_baseline":     True,
    "confidence_level":            0.95,
    "retrievers":                  _RETRIEVERS,
    "chunkers":                    _CHUNKERS,
}

# Quick sanity check: 1 seed, small dataset slice
COLAB_QUICK_CONFIG = {
    **_BASE,
    "seeds": [42],
    "bootstrap_samples": 200,
    "datasets": [
        {"name": "squad_v2",   "split": "validation", "max_examples": 60,
         "candidate_pool_size": 300, "answerable_only": True},
        {"name": "hotpot_qa",  "split": "validation", "max_examples": 30,
         "config": "distractor"},
    ],
}

# Full run: 3 seeds, larger dataset slices
COLAB_FULL_CONFIG = {
    **_BASE,
    "seeds": [13, 21, 34],
    "bootstrap_samples": 1000,
    "datasets": [
        {"name": "squad_v2",   "split": "validation", "max_examples": 400,
         "candidate_pool_size": 2000, "answerable_only": True},
        {"name": "hotpot_qa",  "split": "validation", "max_examples": 150,
         "config": "distractor"},
    ],
}

# ─────────────────────────────────────────────────────────────────────────────
ACTIVE_CONFIG = COLAB_QUICK_CONFIG
# ─────────────────────────────────────────────────────────────────────────────

print(f"Active config: {len(ACTIVE_CONFIG['seeds'])} seed(s), "
      f"model={ACTIVE_CONFIG['generator_model']}")
for ds in ACTIVE_CONFIG['datasets']:
    print(f"  {ds['name']}: {ds['max_examples']} examples")

## Run Experiments

In [ ]:
from pathlib import Path
from chunkrag.pipeline import run_experiment_suite

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

print(f"Starting experiment: {ACTIVE_CONFIG['generator_model']}")
print(f"Output directory:    {output_path}")

summaries = run_experiment_suite(ACTIVE_CONFIG, output_path)

print(f"\nDone. Generated {len(summaries)} summary rows.")
print(f"Aggregate results: {output_path / 'aggregate_results.json'}")

## Results

In [ ]:
import json
import pandas as pd

with open(output_path / "aggregate_results.json") as f:
    df = pd.DataFrame(json.load(f))

SYSTEM_ORDER = [
    "parametric_baseline",
    "fixed_128", "fixed_256", "fixed_512",
    "recursive_256", "sentence_256", "semantic_256",
]
SYSTEM_LABELS = {
    "parametric_baseline": "parametric_only",
    "fixed_128":           "fixed_128",
    "fixed_256":           "fixed_256",
    "fixed_512":           "fixed_512",
    "recursive_256":       "recursive_256",
    "sentence_256":        "sentence_256",
    "semantic_256":        "semantic_256",
}


def make_report_table(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    data = df[df["dataset"] == dataset].copy()
    data = data.loc[data.groupby("chunker")["f1_mean"].idxmax()].copy()

    for col in ["exact_match_mean", "f1_mean", "recall_at_k_mean", "precision_at_k_mean"]:
        if col in data.columns:
            data[col] = (data[col] * 100).round(1)

    if "avg_chunk_tokens_mean" in data.columns:
        data["avg_chunk_tokens_mean"] = data["avg_chunk_tokens_mean"].round(1)
    if "num_chunks_mean" in data.columns:
        data["num_chunks_mean"] = data["num_chunks_mean"].round(0).astype("Int64")

    data["chunker"] = pd.Categorical(data["chunker"], categories=SYSTEM_ORDER, ordered=True)
    data = data.sort_values("chunker")

    table = pd.DataFrame({
        "System":           data["chunker"].map(SYSTEM_LABELS).values,
        "EM":               data.get("exact_match_mean",     pd.Series()).values,
        "F1":               data.get("f1_mean",              pd.Series()).values,
        "Recall@4":         data.get("recall_at_k_mean",     pd.Series()).values,
        "Precision@4":      data.get("precision_at_k_mean",  pd.Series()).values,
        "Avg chunk tokens": data.get("avg_chunk_tokens_mean", pd.Series()).values,
        "# chunks":         data.get("num_chunks_mean",       pd.Series()).values,
    })

    param = table["System"] == "parametric_only"
    table["Avg chunk tokens"] = table["Avg chunk tokens"].astype(object)
    table["# chunks"] = table["# chunks"].astype(object)
    table.loc[param, ["Avg chunk tokens", "# chunks"]] = "—"

    return table.reset_index(drop=True)


table_squad  = make_report_table(df, "squad_v2")
table_hotpot = make_report_table(df, "hotpot_qa")

print("Table 1: SQuAD v2  (dense retriever)")
display(table_squad)

print("\nTable 2: HotpotQA  (dense retriever)")
display(table_hotpot)

In [ ]:
# Save tables to Drive as CSV (alongside the full aggregate_results.json)
table_squad.to_csv(output_path / "table1_squad_v2.csv", index=False)
table_hotpot.to_csv(output_path / "table2_hotpot_qa.csv", index=False)
print(f"Saved to {output_path}:")
print(f"  aggregate_results.json  — full raw results")
print(f"  table1_squad_v2.csv     — Table 1 (SQuAD v2)")
print(f"  table2_hotpot_qa.csv    — Table 2 (HotpotQA)")